# 05c. Векторизованный causal multi-head attention

Лекция: [Векторизованный attention](../../notes/interview-prep/08f-vectorized-attention.md).

Цель: получить то же вычисление, что с ModuleList, без Python-цикла по головам в forward нового слоя. Сегодня только A: общие проекции и разделение голов. Эталон и перенос весов уже готовы. B — attention и обратная сборка, C — эквивалентность результатов и градиентов.

Без bias, dropout и padding. D=8, H=2: head_dim=4. Размеры намеренно отличаются от прошлого примера. Notebook независим от предыдущих.

## Готовый эталон

Это тот же алгоритм отдельных голов из урока 05. Не нужно переписывать его: используем как эталон. Модель не обучена; проверяем эквивалентность вычислений, а не качество языка.

In [1]:
import math

import torch
from torch import nn

torch.manual_seed(42)


class ReferenceHead(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.query_projection = nn.Linear(d_model, head_dim, bias=False)
        self.key_projection = nn.Linear(d_model, head_dim, bias=False)
        self.value_projection = nn.Linear(d_model, head_dim, bias=False)

    def forward(self, x):
        q = self.query_projection(x)
        k = self.key_projection(x)
        v = self.value_projection(x)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        weights = torch.softmax(scores.masked_fill(blocked, float('-inf')), dim=-1)
        return weights @ v


class ReferenceMultiHead(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.heads = nn.ModuleList([
            ReferenceHead(d_model, d_model // num_heads) for _ in range(num_heads)
        ])
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        return self.output_projection(torch.cat([h(x) for h in self.heads], dim=-1))


B, T, D, H = 2, 3, 8, 2
x = torch.randn(B, T, D)
reference = ReferenceMultiHead(D, H)

## A. Общие проекции и ось голов

В новом слое три Linear(D,D). Их выходные строки сгруппированы по головам. Заполни только project_heads: после проекции (B,T,D) нужны reshape до (B,T,H,head_dim) и transpose до (B,H,T,head_dim).

В forward оставлен явный NotImplementedError для этапа B. Сегодня вызываем только project_heads, не весь model(x).

In [2]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # TODO A: каждая строка — проекция, reshape и transpose.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)


## Перенос весов — готовый код

Матрица отдельной головы имеет форму (head_dim,D). Склеиваем H таких матриц по строкам (dim=0), получая (D,D). Так общая проекция действительно вычисляет те же числа, что отдельные головы.

copy_ копирует значения в существующий параметр. no_grad нужен, чтобы перенос параметров не записывался в граф. Выходную проекцию тоже копируем — иначе выходы всей модели сравнивать бессмысленно.

In [3]:
model = VectorizedCausalAttention(D, H)
with torch.no_grad():
    for attr in ['query_projection', 'key_projection', 'value_projection']:
        packed_weight = torch.cat([getattr(head, attr).weight for head in reference.heads], dim=0)
        getattr(model, attr).weight.copy_(packed_weight)
    model.output_projection.weight.copy_(reference.output_projection.weight)

q, k, v = model.project_heads(x)
for result in (q, k, v):
    assert isinstance(result, torch.Tensor), 'Заполни project_heads'
    assert result.shape == (B, H, T, D // H)

# Эталон создаёт ось голов через stack отдельных результатов.
for result, attr in [(q, 'query_projection'), (k, 'key_projection'), (v, 'value_projection')]:
    expected = torch.stack([getattr(head, attr)(x) for head in reference.heads], dim=1)
    torch.testing.assert_close(result, expected)
assert sum(p.numel() for p in model.parameters()) == 4 * D * D == 256
print('Q/K/V совпали с отдельными головами:', q.shape)
print('Параметров:', sum(p.numel() for p in model.parameters()))

Q/K/V совпали с отдельными головами: torch.Size([2, 2, 3, 4])
Параметров: 256


### Остановка A

1. Почему веса голов склеиваем по dim=0, а их выходы в уроке 05 склеивали по dim=-1?
2. Что означает q[0,1,2] и какова форма этого среза?
3. Почему одинакового seed недостаточно для надёжного сравнения разных реализаций, а копирование соответствующих весов решает задачу?

**Мои ответы:**

1. Не знаю
2. Перавя последовательность, вторая голова, третий токен. Размерность  D
3. Не знаю

**Разбор ментора (исходные ответы сохранены):**

project_heads реализован правильно: список из трёх проекций обрабатывается в порядке Q,K,V, затем результаты распаковываются. Проверки совпадения Q/K/V с отдельными головами прошли. Обнаружена отдельная ошибка отступа: def forward сейчас вне класса. Перед частью B нужно сдвинуть его и всё его тело на четыре пробела вправо, чтобы он стал методом VectorizedCausalAttention. Проверки A этого не обнаружили, потому что вызывали project_heads, а не model(x).
1. Weight имеет форму (out_features,in_features). Поэтому матрицы голов (4,8) склеиваем по строкам, dim=0, получая общую (8,8). Это добавляет выходные координаты, сохраняя восемь входных признаков для каждой. Выходы голов имеют форму (B,T,4): их признаки расположены на последней оси, поэтому cat по dim=-1 даёт (B,T,8). Номер оси выбирается по смыслу тензора.
2. Индексы прочитаны верно: первая последовательность, вторая голова, третья позиция. Но это Query одной головы, поэтому форма среза (head_dim,) = (4,), а не (D,) = (8,). Последняя ось не указана и берётся целиком.
3. Seed задаёт воспроизводимую последовательность случайных чисел, а не соответствие параметров разных архитектур. Отдельные головы создают Q0,K0,V0,Q1,K1,V1, а общие проекции — Q_all,K_all,V_all: порядок распределения случайных значений по ролям отличается. Копирование соответствующих весов и выходной проекции делает сравнение именно проверкой одинаковых вычислений, а не разных случайных моделей.


Сохрани и пришли на проверку. Этап B пока не выполняй.

## B. Attention сразу для B и H

Вернись к forward и замени NotImplementedError. Формы:

```text
Q,K,V:        (B,H,T,head_dim)
scores:       (B,H,T,T)
blocked_mask: (T,T) — broadcasting по B и H
weights:      (B,H,T,T)
context_heads:(B,H,T,head_dim)
combined:     (B,T,D)
output:       (B,T,D)
```

Делитель sqrt(self.head_dim). Транспонируем последние две оси K. Softmax по последней оси. Обратная сборка уже знакома. Не используй глобальные B/T при reshape.

In [4]:
for test_B, test_T in [(2, 3), (1, 1), (3, 5)]:
    test_x = torch.randn(test_B, test_T, D)
    result = model(test_x)
    assert result.shape == test_x.shape
    assert torch.isfinite(result).all()
    torch.testing.assert_close(result, reference(test_x), rtol=1e-4, atol=1e-6)
print('Forward совпадает с эталоном для разных B/T: OK')

with torch.no_grad():
    changed = x.clone()
    changed[:, -1] += torch.arange(1, D + 1, dtype=x.dtype, device=x.device)
    torch.testing.assert_close(model(x)[:, :-1], model(changed)[:, :-1])
print('Причинность: OK')

Forward совпадает с эталоном для разных B/T: OK
Причинность: OK


### Остановка B

1. Почему scores имеют форму (B,H,T,T), а не (B,T,T)?
2. Почему делитель sqrt(head_dim), а не sqrt(D)?
3. Что произойдёт с группировкой, если забыть обратный transpose перед reshape?

**Мои ответы:**

TODO

**Ответы и разбор ментора:**

1. Каждая голова вычисляет собственную матрицу сравнений T×T. Поэтому для B последовательностей и H голов scores имеют форму (B,H,T,T). Ось H сохраняет независимость голов.
2. Одно скалярное произведение складывает head_dim произведений координат, а не D. Поэтому используем sqrt(head_dim). Здесь D=8, H=2, head_dim=4, делитель равен 2. Мотивировка через дисперсию опирается на предположения о координатах, а не гарантирует их статистику после обучения.
3. Без обратного transpose прямой reshape может дать нужную форму (B,T,D), но собрать в одном векторе признаки разных позиций. Сначала возвращаем (B,T,H,head_dim), затем объединяем признаки голов для каждой фиксированной позиции.

Разбор исправления: reshape должен брать batch_size и length из текущего x, а D — из self.d_model. Глобальные B=2,T=3 ломали тест входа (1,1,8): 8 элементов нельзя преобразовать в форму (2,3,8), требующую 48. Ученик исправил это; проверки B прошли.

## C. Совпадают ли градиенты?

Готовая дополнительная проверка. Два независимых входных тензора с одинаковыми значениями позволяют сравнить производные по входу. Очищаем параметры обеих моделей и вычисляем один и тот же искусственный loss. Сравниваем градиенты общих матриц с объединёнными градиентами голов.

Здесь нет optimizer.step: веса не меняются. Часть не является обучением языка.

In [5]:
reference.zero_grad(set_to_none=True)
model.zero_grad(set_to_none=True)
x_ref = x.detach().clone().requires_grad_(True)
x_vec = x.detach().clone().requires_grad_(True)
reference(x_ref).square().mean().backward()
model(x_vec).square().mean().backward()

torch.testing.assert_close(x_ref.grad, x_vec.grad, rtol=1e-4, atol=1e-6)
for attr in ['query_projection', 'key_projection', 'value_projection']:
    expected_grad = torch.cat([getattr(head, attr).weight.grad for head in reference.heads], dim=0)
    actual_grad = getattr(model, attr).weight.grad
    assert actual_grad is not None and torch.isfinite(actual_grad).all()
    torch.testing.assert_close(actual_grad, expected_grad, rtol=1e-4, atol=1e-6)
torch.testing.assert_close(model.output_projection.weight.grad,
                           reference.output_projection.weight.grad,
                           rtol=1e-4, atol=1e-6)
print('Градиенты входа, Q/K/V и выходной проекции совпадают: OK')

Градиенты входа, Q/K/V и выходной проекции совпадают: OK


### Остановка C

1. Почему равенства только forward недостаточно для проверки обучаемой реализации?
2. Почему сравниваем градиенты с допуском, а не требуем точного равенства битов?
3. Доказали ли мы ускорение? Какие измерения понадобятся для такого утверждения?

**Мои ответы:**

TODO

После изменения класса нужно заново создать model и выполнить перенос весов. Перед финальной проверкой: Restart Kernel → Run All → сохранить outputs. Скорость не измерялась; векторизация не является обещанием ускорения на любом устройстве. После проверки — Transformer block.

**Ответы и разбор ментора:**

1. Числа forward могут совпасть при неправильном градиентном пути: например, detach сохраняет значения, но обрывает связь с частью параметров. Поэтому сравниваем также градиенты входа и соответствующих матриц, чтобы проверить вычисление, необходимое для обучения.
2. Floating-point операции имеют округление; разные группировки матричных вычислений могут слегка менять результат. assert_close проверяет близость с абсолютным и относительным допусками, а не равенство битов. Слишком большой допуск может скрыть ошибку.
3. Нет, время не измеряли. Для заявления об ускорении нужны сравнимые устройства, dtype, размеры B/T/D/H, режим forward/backward, прогрев и несколько измерений; для GPU — корректная синхронизация. Уменьшение числа Python-вызовов само по себе не доказывает ускорение на любой задаче.

Итог: по сохранённым outputs прошли проекции, forward на разных размерах, причинность и совпадение градиентов входа и параметров. Это проверка двух реализаций на тестовых входах, не обучение языковой модели. Исходные ответы ученика сохранены, готовые объяснения добавлены ментором.